In [ ]:
# gpu_cps_threat_app.py
# Single-file baseline: GUI + GPU-optimized training + iterative sweeps + charts + CSV exports.
# Requirements (conda/pip):
#   pip install pyside6 pandas pyarrow numpy optuna torch matplotlib scikit-learn
#
# Run:
#   python gpu_cps_threat_app.py

import os
import sys
import json
import math
import time
import traceback
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import optuna

import matplotlib
matplotlib.use("Agg")  # for headless save; GUI displays via image load
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, f1_score

from PySide6 import QtCore, QtWidgets, QtGui


# -------------------------------
# Config / Utilities
# -------------------------------

@dataclass
class AppConfig:
    window_len: int = 120          # seconds at 1 Hz (adjust for your CPS)
    window_stride: int = 10        # step between windows
    max_rows_per_file: Optional[int] = None  # set for dev speed, e.g., 200_000
    num_workers: int = max(2, os.cpu_count() // 2)
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42


def set_reproducible(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def safe_mkdir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def now_ms() -> int:
    return int(time.time() * 1000)


# -------------------------------
# MITRE model + Timeline ingestion
# -------------------------------

def load_mitre_model(mitre_path: str) -> Dict:
    """
    Accepts JSON threat model (MITRE ATT&CK export or your own schema).
    In baseline we only need a dictionary; you will extend mapping logic later.
    """
    if not mitre_path:
        return {}
    with open(mitre_path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_timeline(timeline_path: str) -> pd.DataFrame:
    """
    Expect columns: start_time, end_time, threat_type (optional technique_id).
    """
    df = pd.read_csv(timeline_path)
    for c in ["start_time", "end_time"]:
        df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")
    if "threat_type" not in df.columns:
        raise ValueError("Timeline CSV must include 'threat_type'.")
    if "technique_id" not in df.columns:
        df["technique_id"] = ""
    return df.dropna(subset=["start_time", "end_time"])


def apply_labels_from_timeline(
    telemetry: pd.DataFrame,
    timeline: pd.DataFrame,
    timestamp_col: str = "timestamp",
    label_col: str = "label",
    threat_col: str = "threat_type",
    technique_col: str = "technique_id",
) -> pd.DataFrame:
    """
    Produces per-row binary label plus threat_type/technique_id for positive rows.
    If overlapping multiple events, uses the first match (you can refine).
    """
    df = telemetry.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], utc=True, errors="coerce")
    df = df.dropna(subset=[timestamp_col]).sort_values(timestamp_col).reset_index(drop=True)

    df[label_col] = 0
    df[threat_col] = ""
    df[technique_col] = ""

    # Efficient interval labeling: iterate events; for large data use interval trees later.
    ts = df[timestamp_col].values.astype("datetime64[ns]")
    for _, ev in timeline.iterrows():
        s = ev["start_time"].to_datetime64()
        e = ev["end_time"].to_datetime64()
        mask = (ts >= s) & (ts <= e)
        if mask.any():
            df.loc[mask, label_col] = 1
            # Assign threat/tech only if empty to keep first match
            empty = df.loc[mask, threat_col].eq("")
            df.loc[mask & empty.values, threat_col] = str(ev["threat_type"])
            df.loc[mask & empty.values, technique_col] = str(ev["technique_id"])
    return df


# -------------------------------
# Dataset builders (tabular + temporal windows)
# -------------------------------

def infer_feature_columns(df: pd.DataFrame) -> List[str]:
    drop = {"timestamp", "label", "threat_type", "technique_id"}
    cols = []
    for c in df.columns:
        if c in drop:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    if not cols:
        raise ValueError("No numeric feature columns found (after excluding timestamp/label/threat_type/technique_id).")
    return cols


class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


class WindowDataset(Dataset):
    """
    Builds windows from continuous telemetry arrays. Label of a window = max(label) in window.
    """
    def __init__(self, X: np.ndarray, y: np.ndarray, window_len: int, stride: int):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.int64)
        self.window_len = int(window_len)
        self.stride = int(stride)

        n = len(self.X)
        self.starts = list(range(0, max(0, n - self.window_len + 1), self.stride))

    def __len__(self) -> int:
        return len(self.starts)

    def __getitem__(self, idx: int):
        s = self.starts[idx]
        e = s + self.window_len
        xw = torch.from_numpy(self.X[s:e]).T  # shape: [features, time]
        yw = int(self.y[s:e].max())
        return xw, torch.tensor(yw, dtype=torch.long)


# -------------------------------
# Models (Technique A: MLP, Technique B: 1D CNN)
# -------------------------------

class MLPClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden: int, depth: int, dropout: float):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout)]
            d = hidden
        layers += [nn.Linear(d, 2)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class CNN1DClassifier(nn.Module):
    """
    Input: [B, C, T]
    """
    def __init__(self, channels: int, base: int, layers: int, dropout: float):
        super().__init__()
        mods = []
        c_in = channels
        c = base
        for i in range(layers):
            mods += [
                nn.Conv1d(c_in, c, kernel_size=5, padding=2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.MaxPool1d(kernel_size=2),
            ]
            c_in = c
            c = min(c * 2, 512)
        self.backbone = nn.Sequential(*mods)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(c_in, 2),
        )

    def forward(self, x):
        z = self.backbone(x)
        return self.head(z)


# -------------------------------
# Training / Evaluation (GPU-optimized)
# -------------------------------

def make_loader(ds: Dataset, batch: int, shuffle: bool, cfg: AppConfig) -> DataLoader:
    return DataLoader(
        ds,
        batch_size=batch,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=(cfg.device == "cuda"),
        persistent_workers=(cfg.num_workers > 0),
        prefetch_factor=2 if cfg.num_workers > 0 else None,
        drop_last=False,
    )


@torch.no_grad()
def eval_model(model: nn.Module, loader: DataLoader, device: str) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        logits = model(xb)
        pred = torch.argmax(logits, dim=1)
        ys.append(yb.detach().cpu().numpy())
        ps.append(pred.detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: str,
    lr: float,
    weight_decay: float,
    epochs: int,
    amp: bool = True,
) -> Dict:
    model.to(device)
    model.train()

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(amp and device == "cuda"))

    best_f1 = -1.0
    best_state = None

    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(amp and device == "cuda")):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        # simple val checkpoint on F1
        y_true, y_pred = eval_model(model, val_loader, device)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return {"best_f1": best_f1}


def compute_confusion(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, int]:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)}


# -------------------------------
# Sweep Orchestrator
# -------------------------------

def split_train_val(X: np.ndarray, y: np.ndarray, val_frac: float = 0.2) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = len(X)
    idx = np.arange(n)
    np.random.shuffle(idx)
    cut = int(n * (1 - val_frac))
    tr, va = idx[:cut], idx[cut:]
    return X[tr], y[tr], X[va], y[va]


def run_sweeps(
    mitre_path: str,
    timeline_path: str,
    train_paths: List[str],
    test_path: str,
    out_dir: str,
    sweeps: int,
    cfg: AppConfig,
    progress_cb=None,
) -> None:
    safe_mkdir(out_dir)
    set_reproducible(cfg.seed)

    mitre = load_mitre_model(mitre_path)  # reserved for your later mapping logic
    timeline = load_timeline(timeline_path)

    # ---- Load + label training data (concatenate multiple CSVs)
    train_frames = []
    for p in train_paths:
        df = pd.read_csv(p, engine="pyarrow")
        if cfg.max_rows_per_file:
            df = df.head(cfg.max_rows_per_file)
        df = apply_labels_from_timeline(df, timeline)
        train_frames.append(df)
    train_df = pd.concat(train_frames, ignore_index=True)

    # ---- Load + label test data
    test_df = pd.read_csv(test_path, engine="pyarrow")
    if cfg.max_rows_per_file:
        test_df = test_df.head(cfg.max_rows_per_file)
    test_df = apply_labels_from_timeline(test_df, timeline)

    feat_cols = infer_feature_columns(train_df)
    X_train_full = train_df[feat_cols].to_numpy(dtype=np.float32, copy=True)
    y_train_full = train_df["label"].to_numpy(dtype=np.int64, copy=True)

    X_test_full = test_df[feat_cols].to_numpy(dtype=np.float32, copy=True)
    y_test_full = test_df["label"].to_numpy(dtype=np.int64, copy=True)

    # Normalize (simple z-score from training)
    mu = X_train_full.mean(axis=0)
    sig = X_train_full.std(axis=0) + 1e-6
    X_train_full = (X_train_full - mu) / sig
    X_test_full = (X_test_full - mu) / sig

    # Build datasets
    Xtr, ytr, Xva, yva = split_train_val(X_train_full, y_train_full, val_frac=0.2)

    # Tabular datasets
    ds_tr_tab = TabularDataset(Xtr, ytr)
    ds_va_tab = TabularDataset(Xva, yva)
    ds_te_tab = TabularDataset(X_test_full, y_test_full)

    # Window datasets (temporal)
    ds_tr_win = WindowDataset(Xtr, ytr, cfg.window_len, cfg.window_stride)
    ds_va_win = WindowDataset(Xva, yva, cfg.window_len, cfg.window_stride)
    ds_te_win = WindowDataset(X_test_full, y_test_full, cfg.window_len, cfg.window_stride)

    # Tracking outputs
    confusion_rows = []
    ablation_rows = []  # one row per iter per technique
    trials_rows = []

    def log(msg: str):
        if progress_cb:
            progress_cb(msg)

    # ---- Optuna objective (evaluates multiple techniques per trial)
    def objective(trial: optuna.Trial) -> float:
        # Shared hyperparams
        lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        wd = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        epochs = trial.suggest_int("epochs", 3, 10)
        batch = trial.suggest_categorical("batch", [256, 512, 1024, 2048])
        amp = True

        # Technique A (MLP)
        mlp_hidden = trial.suggest_int("mlp_hidden", 64, 512, log=True)
        mlp_depth = trial.suggest_int("mlp_depth", 1, 5)
        mlp_dropout = trial.suggest_float("mlp_dropout", 0.0, 0.5)

        # Technique B (CNN)
        cnn_base = trial.suggest_int("cnn_base", 16, 128, log=True)
        cnn_layers = trial.suggest_int("cnn_layers", 1, 4)
        cnn_dropout = trial.suggest_float("cnn_dropout", 0.0, 0.5)

        # DataLoaders
        tr_tab = make_loader(ds_tr_tab, batch=batch, shuffle=True, cfg=cfg)
        va_tab = make_loader(ds_va_tab, batch=batch, shuffle=False, cfg=cfg)
        te_tab = make_loader(ds_te_tab, batch=batch, shuffle=False, cfg=cfg)

        # For windows, batch must be smaller (memory); derive conservative batch
        wbatch = max(32, batch // 32)
        tr_win = make_loader(ds_tr_win, batch=wbatch, shuffle=True, cfg=cfg)
        va_win = make_loader(ds_va_win, batch=wbatch, shuffle=False, cfg=cfg)
        te_win = make_loader(ds_te_win, batch=wbatch, shuffle=False, cfg=cfg)

        results = {}

        # ---- Technique A: MLP
        model_a = MLPClassifier(in_dim=len(feat_cols), hidden=mlp_hidden, depth=mlp_depth, dropout=mlp_dropout)
        a_stats = train_model(model_a, tr_tab, va_tab, cfg.device, lr, wd, epochs, amp=amp)
        y_true_a, y_pred_a = eval_model(model_a, te_tab, cfg.device)
        f1_a = f1_score(y_true_a, y_pred_a, zero_division=0)
        conf_a = compute_confusion(y_true_a, y_pred_a)
        results["MLP"] = {"f1": f1_a, **conf_a}

        # ---- Technique B: CNN1D
        model_b = CNN1DClassifier(channels=len(feat_cols), base=cnn_base, layers=cnn_layers, dropout=cnn_dropout)
        b_stats = train_model(model_b, tr_win, va_win, cfg.device, lr, wd, epochs, amp=amp)
        y_true_b, y_pred_b = eval_model(model_b, te_win, cfg.device)
        f1_b = f1_score(y_true_b, y_pred_b, zero_division=0)
        conf_b = compute_confusion(y_true_b, y_pred_b)
        results["CNN1D"] = {"f1": f1_b, **conf_b}

        # Record per-technique metrics in trial user attrs
        trial.set_user_attr("results", results)

        # Choose "best technique" for the trial by a cost-aware score
        # You can tune these weights to your operational risk tolerance.
        def score(r):
            # maximize F1; penalize FN more than FP (common in threat detection)
            return (2.0 * r["f1"]) - (0.0005 * r["FP"]) - (0.0010 * r["FN"])

        best_tech = max(results.keys(), key=lambda k: score(results[k]))
        trial.set_user_attr("best_technique", best_tech)

        # Objective value
        return score(results[best_tech])

    log(f"Device: {cfg.device} | CPU workers: {cfg.num_workers} | Sweeps: {sweeps}")

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=cfg.seed))
    study.optimize(objective, n_trials=sweeps)

    # Build iteration-wise outputs (best technique PER TRIAL)
    for i, t in enumerate(study.trials, start=1):
        if t.value is None:
            continue
        res = t.user_attrs.get("results", {})
        best_tech = t.user_attrs.get("best_technique", None)
        if not res or not best_tech:
            continue

        # per-technique ablation row
        for tech, r in res.items():
            ablation_rows.append({
                "iteration": i,
                "technique": tech,
                "F1": float(r["f1"]),
                "TP": int(r["TP"]), "FP": int(r["FP"]), "TN": int(r["TN"]), "FN": int(r["FN"]),
            })

        # confusion evolution row (best of this iteration)
        rbest = res[best_tech]
        confusion_rows.append({
            "iteration": i,
            "best_technique": best_tech,
            "TP": int(rbest["TP"]), "FP": int(rbest["FP"]), "TN": int(rbest["TN"]), "FN": int(rbest["FN"]),
            "F1": float(rbest["f1"]),
            "objective": float(t.value),
        })

        # raw trial params
        tr = {"iteration": i, "objective": float(t.value), "best_technique": best_tech}
        tr.update(t.params)
        trials_rows.append(tr)

    # ---- Export CSVs
    df_conf = pd.DataFrame(confusion_rows)
    df_abla = pd.DataFrame(ablation_rows)
    df_trials = pd.DataFrame(trials_rows)

    conf_csv = os.path.join(out_dir, "confusion_by_iter.csv")
    abla_csv = os.path.join(out_dir, "ablation_f1_by_iter.csv")
    trials_csv = os.path.join(out_dir, "trials_raw.csv")

    df_conf.to_csv(conf_csv, index=False)
    df_abla.to_csv(abla_csv, index=False)
    df_trials.to_csv(trials_csv, index=False)

    # ---- Charts
    # Confusion evolution chart (best technique per iteration)
    fig1 = plt.figure()
    plt.plot(df_conf["iteration"], df_conf["TP"], label="TP")
    plt.plot(df_conf["iteration"], df_conf["FP"], label="FP")
    plt.plot(df_conf["iteration"], df_conf["TN"], label="TN")
    plt.plot(df_conf["iteration"], df_conf["FN"], label="FN")
    plt.xlabel("Test Iteration")
    plt.ylabel("Number of True/Predicted Events")
    plt.title("Confusion Matrix Elements vs Iteration (Best Outcome per Sweep)")
    plt.legend()
    conf_png = os.path.join(out_dir, "confusion_evolution.png")
    plt.tight_layout()
    plt.savefig(conf_png, dpi=200)
    plt.close(fig1)

    # Ablation chart: F1 per technique per iteration
    fig2 = plt.figure()
    for tech in sorted(df_abla["technique"].unique()):
        sub = df_abla[df_abla["technique"] == tech].sort_values("iteration")
        plt.plot(sub["iteration"], sub["F1"], label=tech)
    plt.xlabel("Test Iteration")
    plt.ylabel("F1 Score")
    plt.title("Ablation: F1 by Technique vs Iteration")
    plt.legend()
    abla_png = os.path.join(out_dir, "ablation_f1.png")
    plt.tight_layout()
    plt.savefig(abla_png, dpi=200)
    plt.close(fig2)

    # Save summary text
    summary_path = os.path.join(out_dir, "summary.txt")
    best = study.best_trial
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(f"Device: {cfg.device}\n")
        f.write(f"Best objective: {best.value}\n")
        f.write(f"Best technique: {best.user_attrs.get('best_technique')}\n")
        f.write(f"Best params: {best.params}\n")

    log("Done.")
    log(f"Saved:\n- {conf_csv}\n- {abla_csv}\n- {trials_csv}\n- {conf_png}\n- {abla_png}\n- {summary_path}")


# -------------------------------
# GUI (PySide6)
# -------------------------------

class Worker(QtCore.QThread):
    message = QtCore.Signal(str)
    finished_ok = QtCore.Signal(str)
    finished_err = QtCore.Signal(str)

    def __init__(self, args: dict):
        super().__init__()
        self.args = args

    def run(self):
        try:
            def cb(msg: str):
                self.message.emit(msg)

            cfg = AppConfig(
                window_len=self.args["window_len"],
                window_stride=self.args["window_stride"],
                max_rows_per_file=self.args["max_rows_per_file"],
                num_workers=self.args["num_workers"],
                seed=42,
            )
            run_sweeps(
                mitre_path=self.args["mitre_path"],
                timeline_path=self.args["timeline_path"],
                train_paths=self.args["train_paths"],
                test_path=self.args["test_path"],
                out_dir=self.args["out_dir"],
                sweeps=self.args["sweeps"],
                cfg=cfg,
                progress_cb=cb,
            )
            self.finished_ok.emit(self.args["out_dir"])
        except Exception:
            self.finished_err.emit(traceback.format_exc())


class MainWindow(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("GPU CPS Threat Detection Sweeper (Baseline)")
        self.resize(1000, 720)

        w = QtWidgets.QWidget()
        self.setCentralWidget(w)
        layout = QtWidgets.QVBoxLayout(w)

        # File pickers
        self.mitre_path = self._file_picker(layout, "MITRE Threat Model (JSON):", filt="JSON (*.json);;All Files (*)")
        self.timeline_path = self._file_picker(layout, "Red Team Timeline (CSV):", filt="CSV (*.csv);;All Files (*)")
        self.train_paths = self._multi_file_picker(layout, "Training CSVs (multi-select):", filt="CSV (*.csv);;All Files (*)")
        self.test_path = self._file_picker(layout, "Test CSV (single):", filt="CSV (*.csv);;All Files (*)")
        self.out_dir = self._dir_picker(layout, "Output Folder:")

        # Controls
        form = QtWidgets.QFormLayout()
        layout.addLayout(form)

        self.sweeps_spin = QtWidgets.QSpinBox()
        self.sweeps_spin.setRange(16, 1000)
        self.sweeps_spin.setValue(16)
        form.addRow("Sweep iterations (>=16):", self.sweeps_spin)

        self.window_len_spin = QtWidgets.QSpinBox()
        self.window_len_spin.setRange(10, 3600)
        self.window_len_spin.setValue(120)
        form.addRow("Window length (seconds):", self.window_len_spin)

        self.window_stride_spin = QtWidgets.QSpinBox()
        self.window_stride_spin.setRange(1, 3600)
        self.window_stride_spin.setValue(10)
        form.addRow("Window stride (seconds):", self.window_stride_spin)

        self.max_rows_spin = QtWidgets.QSpinBox()
        self.max_rows_spin.setRange(0, 10_000_000)
        self.max_rows_spin.setValue(0)
        self.max_rows_spin.setToolTip("0 = no limit (use full files). For dev, set e.g., 200000.")
        form.addRow("Max rows per file (0=all):", self.max_rows_spin)

        self.workers_spin = QtWidgets.QSpinBox()
        self.workers_spin.setRange(0, max(1, os.cpu_count() or 1))
        self.workers_spin.setValue(max(2, (os.cpu_count() or 8) // 2))
        form.addRow("CPU DataLoader workers:", self.workers_spin)

        # Run button + status
        self.run_btn = QtWidgets.QPushButton("Run Sweeps")
        self.run_btn.clicked.connect(self.on_run)
        layout.addWidget(self.run_btn)

        self.log = QtWidgets.QPlainTextEdit()
        self.log.setReadOnly(True)
        layout.addWidget(self.log, stretch=1)

        # Preview images
        img_layout = QtWidgets.QHBoxLayout()
        layout.addLayout(img_layout)
        self.conf_img = QtWidgets.QLabel("Confusion evolution chart will appear here.")
        self.conf_img.setAlignment(QtCore.Qt.AlignCenter)
        self.conf_img.setMinimumHeight(220)
        img_layout.addWidget(self.conf_img, stretch=1)

        self.abla_img = QtWidgets.QLabel("Ablation F1 chart will appear here.")
        self.abla_img.setAlignment(QtCore.Qt.AlignCenter)
        self.abla_img.setMinimumHeight(220)
        img_layout.addWidget(self.abla_img, stretch=1)

        self.worker = None

    def _file_picker(self, parent_layout, label, filt="All Files (*)"):
        h = QtWidgets.QHBoxLayout()
        parent_layout.addLayout(h)
        h.addWidget(QtWidgets.QLabel(label))
        le = QtWidgets.QLineEdit()
        h.addWidget(le, stretch=1)
        btn = QtWidgets.QPushButton("Browse")
        h.addWidget(btn)

        def pick():
            path, _ = QtWidgets.QFileDialog.getOpenFileName(self, "Select file", "", filt)
            if path:
                le.setText(path)
        btn.clicked.connect(pick)
        return le

    def _multi_file_picker(self, parent_layout, label, filt="All Files (*)"):
        v = QtWidgets.QVBoxLayout()
        parent_layout.addLayout(v)
        v.addWidget(QtWidgets.QLabel(label))
        listw = QtWidgets.QListWidget()
        v.addWidget(listw, stretch=1)
        h = QtWidgets.QHBoxLayout()
        v.addLayout(h)
        add_btn = QtWidgets.QPushButton("Add")
        clr_btn = QtWidgets.QPushButton("Clear")
        h.addWidget(add_btn)
        h.addWidget(clr_btn)

        def add():
            paths, _ = QtWidgets.QFileDialog.getOpenFileNames(self, "Select training CSVs", "", filt)
            for p in paths:
                listw.addItem(p)

        def clear():
            listw.clear()

        add_btn.clicked.connect(add)
        clr_btn.clicked.connect(clear)
        return listw

    def _dir_picker(self, parent_layout, label):
        h = QtWidgets.QHBoxLayout()
        parent_layout.addLayout(h)
        h.addWidget(QtWidgets.QLabel(label))
        le = QtWidgets.QLineEdit()
        h.addWidget(le, stretch=1)
        btn = QtWidgets.QPushButton("Browse")
        h.addWidget(btn)

        def pick():
            path = QtWidgets.QFileDialog.getExistingDirectory(self, "Select output directory")
            if path:
                le.setText(path)
        btn.clicked.connect(pick)
        return le

    def append_log(self, msg: str):
        self.log.appendPlainText(msg)
        self.log.verticalScrollBar().setValue(self.log.verticalScrollBar().maximum())

    def on_run(self):
        mitre = self.mitre_path.text().strip()
        timeline = self.timeline_path.text().strip()
        test = self.test_path.text().strip()
        out_dir = self.out_dir.text().strip()

        train_paths = [self.train_paths.item(i).text() for i in range(self.train_paths.count())]

        if not timeline or not train_paths or not test or not out_dir:
            QtWidgets.QMessageBox.critical(self, "Missing input", "Provide: timeline CSV, >=1 training CSV, test CSV, and output folder.")
            return

        sweeps = int(self.sweeps_spin.value())
        window_len = int(self.window_len_spin.value())
        window_stride = int(self.window_stride_spin.value())
        max_rows = int(self.max_rows_spin.value())
        max_rows = None if max_rows == 0 else max_rows
        workers = int(self.workers_spin.value())

        safe_mkdir(out_dir)

        self.run_btn.setEnabled(False)
        self.append_log(f"Starting sweeps in: {out_dir}")

        args = dict(
            mitre_path=mitre,
            timeline_path=timeline,
            train_paths=train_paths,
            test_path=test,
            out_dir=out_dir,
            sweeps=sweeps,
            window_len=window_len,
            window_stride=window_stride,
            max_rows_per_file=max_rows,
            num_workers=workers,
        )

        self.worker = Worker(args)
        self.worker.message.connect(self.append_log)
        self.worker.finished_ok.connect(self.on_done_ok)
        self.worker.finished_err.connect(self.on_done_err)
        self.worker.start()

    def on_done_ok(self, out_dir: str):
        self.run_btn.setEnabled(True)
        self.append_log("Completed successfully.")
        self._load_images(out_dir)

    def on_done_err(self, tb: str):
        self.run_btn.setEnabled(True)
        self.append_log("ERROR:")
        self.append_log(tb)
        QtWidgets.QMessageBox.critical(self, "Run failed", "See log for stack trace.")

    def _load_images(self, out_dir: str):
        conf_png = os.path.join(out_dir, "confusion_evolution.png")
        abla_png = os.path.join(out_dir, "ablation_f1.png")

        def set_pix(label: QtWidgets.QLabel, path: str):
            if os.path.exists(path):
                pix = QtGui.QPixmap(path)
                label.setPixmap(pix.scaled(label.size(), QtCore.Qt.KeepAspectRatio, QtCore.Qt.SmoothTransformation))
            else:
                label.setText(f"Not found:\n{path}")

        set_pix(self.conf_img, conf_png)
        set_pix(self.abla_img, abla_png)

    def resizeEvent(self, event):
        super().resizeEvent(event)
        out_dir = self.out_dir.text().strip()
        if out_dir:
            self._load_images(out_dir)


def main():
    app = QtWidgets.QApplication(sys.argv)
    w = MainWindow()
    w.show()
    sys.exit(app.exec())


if __name__ == "__main__":
    main()
